In [6]:
import json
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI()

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

In [7]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {"type": "integer", "description": "The movie ID"}
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew credits for a movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {"type": "integer", "description": "The movie ID"}
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get a list of similar movies for a movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {"type": "integer", "description": "The movie ID"}
                },
                "required": ["movie_id"],
            },
        },
    },
]

In [8]:
def call_tool(name, args):
    if name == "get_popular_movies":
        url = f"{BASE_URL}/movies"
    elif name == "get_movie_details":
        url = f"{BASE_URL}/movies/{args['movie_id']}"
    elif name == "get_movie_credits":
        url = f"{BASE_URL}/movies/{args['movie_id']}/credits"
    elif name == "get_similar_movies":
        url = f"{BASE_URL}/movies/{args['movie_id']}/similar"
    else:
        return json.dumps({"error": f"Unknown tool: {name}"})
    return requests.get(url).text

In [9]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a movie expert agent.\n\n"
            "Rules:\n"
            "- Always use the provided tools to look up real movie data. Never make up information.\n"
            "- When listing movies, always include their ID (e.g. '듄: 파트 2 (ID: 693134)').\n"
            "- When the user refers to a movie by name from earlier in the conversation, use the ID from that context to call tools.\n"
            "- When the user asks for similar movies or more details, infer which movie they mean from conversation history.\n"
            "- Remember the user's favorite genres and watched movies from the conversation.\n"
            "- NEVER recommend a movie the user said they already watched.\n"
            "- Respond in the same language the user uses."
        ),
    },
]


def chat():
    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
        )
        msg = response.choices[0].message
        messages.append(msg)

        if msg.tool_calls:
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"[Tool Call] {tc.function.name}({args})")
                result = call_tool(tc.function.name, args)
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": result,
                    }
                )
        else:
            messages.append({"role": "assistant", "content": msg.content})
            print(f"AI: {msg.content}")
            return

In [10]:
while True:
    message = input("You: ")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        chat()

User: 지금 인기있는 영화 알려주세요
[Tool Call] get_popular_movies({})
AI: 현재 인기 있는 영화 목록입니다:

1. **Mercy** (ID: 1236153)
   - 개요: 미래의 탐정이 아내를 살해한 혐의로 재판을 받고, 자신이 지지했던 고급 AI 판사 앞에서 무죄를 입증해야 하는 이야기입니다.
   - 개봉일: 2026-01-20
   - 평점: 7.1

   ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **Shelter** (ID: 1290821)
   - 개요: 외딴 섬에서 자가 망명 중인 남자가 폭풍우에서 젊은 소녀를 구하게 되어 과거의 적들로부터 그녀를 보호해야 하는 이야기를 그립니다.
   - 개봉일: 2026-01-28
   - 평점: 7.0

   ![Shelter](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

3. **Les Orphelins** (ID: 1244975)
   - 개요: 고아 출신의 두 친구가 첫사랑의 딸을 구하기 위해 팀을 이뤄 사건의 배후를 파헤치는 이야기를 다룹니다.
   - 개봉일: 2025-08-20
   - 평점: 6.1

   ![Les Orphelins](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff** (ID: 799882)
   - 개요: 외딴 섬에 살고 있는 전 해적 여성이 복수심 가득한 전 선장이 돌아오며 자신의 과거와 맞서 싸우는 이야기입니다.
   - 개봉일: 2026-02-17
   - 평점: 5.6

   ![The Bluff](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

5. **A Woman Scorned** (ID: 130